# 使用 Meta 家族模型進行建構

## 介紹

本課程將涵蓋：

- 探索兩個主要的 Meta 家族模型 - Llama 3.1 和 Llama 3.2
- 了解每個模型的使用案例和應用場景
- 程式碼範例展示每個模型的獨特功能


## Meta 家族模型

在本課程中，我們將探討來自 Meta 家族或「Llama 群」的兩個模型 - Llama 3.1 和 Llama 3.2

這些模型有不同的變體，並可在 [Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) 使用。

> **注意：** GitHub Models 將於 2026 年 7 月底退役。這裡有更多關於使用 [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) 進行 AI 模型原型設計的詳情。

模型變體：
- Llama 3.1 - 70B Instruct
- Llama 3.1 - 405B Instruct
- Llama 3.2 - 11B Vision Instruct
- Llama 3.2 - 90B Vision Instruct

*注意：Llama 3 亦可在 Microsoft Foundry Models 中使用，但本課程不涵蓋*



## Llama 3.1 

Llama 3.1 擁有 4050 億參數，屬於開源大型語言模型（LLM）範疇。 

此版本為先前發佈的 Llama 3 的升級，提供了： 

- 更大的上下文視窗 - 128k 代幣 對比 8k 代幣 
- 更大的最大輸出代幣數 - 4096 對比 2048 
- 更佳的多語言支援 - 由於訓練代幣數增加 

這些使 Llama 3.1 能夠處理更複雜的用例，建構生成式 AI 應用，包括： 
- 原生函數調用 - 可調用大型語言模型工作流之外的外部工具與函數
- 更佳的檢索增強生成（RAG）效能 - 由於更大的上下文視窗 
- 合成數據生成 - 能夠為微調等任務創建有效數據 



### 原生函數呼叫 

Llama 3.1 已經過微調，使其在進行函數或工具呼叫方面更有效。它還有兩個內建的工具，模型可以根據使用者的提示識別需要使用這些工具。這些工具為： 

- **Brave Search** - 可用於通過網絡搜尋獲取最新資訊，例如天氣 
- **Wolfram Alpha** - 可用於更複雜的數學運算，無需自行撰寫函數。 

你也可以創建自訂工具，讓 LLM 進行呼叫。 

在以下程式碼範例中： 

- 我們在系統提示中定義可用工具（brave_search、wolfram_alpha）。 
- 傳送一個使用者提示，詢問某個城市的天氣。 
- LLM 將回應一個呼叫 Brave Search 工具的指令，形式如下 `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*注意：此範例只進行工具呼叫，若你想取得結果，需在 Brave API 頁面創建免費帳號並定義該函數本身` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

儘管 Llama 3.1 是一個大型語言模型，但它的一個限制是多模態。也就是說，能夠使用不同類型的輸入，例如圖像作為提示並提供回應。這項能力是 Llama 3.2 的主要特點之一。這些特點還包括：

- 多模態 - 具備評估文本和圖像提示的能力
- 小型至中型尺寸變體（11B 和 90B）- 提供靈活的部署選項，
- 純文本變體（1B 和 3B）- 允許模型部署在邊緣/行動裝置，並提供低延遲

多模態支持代表了開源模型世界中的一大進展。以下代碼範例同時使用圖像和文本提示，從 Llama 3.2 90B 獲取圖像分析結果。

### Llama 3.2 的多模態支持


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## 學習不止於此，繼續旅程

完成本課程後，請查看我們的[生成式 AI 學習合集](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst)，繼續提升你的生成式 AI 知識！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
